In [18]:
import torch
from torch import nn
import numpy as np
import pandas as pd
import dataclasses
from dataclasses import dataclass, field
from typing import List
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from template_modules import EncoderStaticBase, EncoderStaticBaseConfig
from collections.abc import Iterable
from basic_conv1d import bn_drop_lin
import warnings
warnings.filterwarnings('ignore')
from clinical_ts.template_modules import ShapeConfig

In [ ]:
# ----------------------------------------
# Config (matches YAML)
# ----------------------------------------
BATCH_SIZE = 32
EPOCHS     = 40
LR         = 0.001
WEIGHT_DECAY = 0.001
DROPOUT    = 0.5
LIN_FTRS   = [128, 128, 128]

In [ ]:
# ----------------------------------------
# 1. Load data (same as multimodal.py)
# ----------------------------------------
print("Loading data...")
df = pd.read_csv('/fs/dss/home/gaad2403/MDS-ED/src/data/memmap/mds_ed.csv',
                 low_memory=False)
print(f"shape: {df.shape}")

In [ ]:
input_cols = [c for c in df.columns if c.split("_")[0] in ['biometrics','demographics','labvalues','vitals']]

# median imputation based on training set
df_train = df[df['general_strat_fold'] < 18]
train_medians = df_train[input_cols].median().to_dict()
train_nans = [c for c,v in df_train[input_cols].isna().sum().to_dict().items() if v > 0]
for c in train_nans:
    df.loc[df[c].isna(), c] = train_medians[c]
df = df.copy()

In [ ]:
# categorical vs continuous feature split (same as multimodal.py)
unique_counts = {c: len(np.unique(np.array(df[c]))) for c in input_cols}
cat_features = [c for c,v in unique_counts.items() if v < 10 and not c.endswith("nan") and not c.startswith("labvalues")]
cat_features_dim = [unique_counts[c] for c in cat_features]
cont_features = [c for c in input_cols if c not in cat_features]

print(f"categorical features: {len(cat_features)} dimensions: {cat_features_dim} continuous features: {len(cont_features)}")

In [ ]:
# ----------------------------------------
# 3. Train/Val/Test split
# ----------------------------------------
train_df = df[df['general_strat_fold'].isin(range(0, 18))].reset_index(drop=True)
val_df   = df[df['general_strat_fold'] == 18].reset_index(drop=True)
test_df  = df[df['general_strat_fold'] == 19].reset_index(drop=True)

# only first ECG per stay for val/test
val_df   = val_df[val_df['general_ecg_no_within_stay'] == 0].reset_index(drop=True)
test_df  = test_df[test_df['general_ecg_no_within_stay'] == 0].reset_index(drop=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# ----------------------------------------
# 4. Dataset
# ----------------------------------------
class TabularDataset(Dataset):
    def __init__(self, df, cont_features, cat_features, lbl_cols):

        # Continuous features
        self.cont = torch.tensor(
            df[cont_features].values,
            dtype=torch.float32
        )

        # Categorical features
        self.cat = torch.tensor(
            df[cat_features].values,
            dtype=torch.long
        )

        # Labels (FIXED)
        self.labels = torch.tensor(
            df[lbl_cols].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.cont)

    def __getitem__(self, idx):
        return self.cont[idx], self.cat[idx], self.labels[idx]

train_ds = TabularDataset(train_df, cont_features, cat_features, target_cols)
val_ds   = TabularDataset(val_df,   cont_features, cat_features, target_cols)
test_ds  = TabularDataset(test_df,  cont_features, cat_features, target_cols)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


In [ ]:
class BasicEncoderStatic(EncoderStaticBase):
    def __init__(self, hparams_encoder_static, hparams_input_shape, target_dim=None):
        super().__init__(hparams_encoder_static, hparams_input_shape, target_dim)
        self.input_channels_cat = hparams_input_shape.static_dim_cat
        self.input_channels_cont = hparams_input_shape.static_dim
        assert(len(hparams_encoder_static.embedding_dims)==hparams_input_shape.static_dim_cat and len(hparams_encoder_static.vocab_sizes)==hparams_input_shape.static_dim_cat)
        self.embeddings = nn.ModuleList() if hparams_input_shape.static_dim_cat is not None else None
        for v,e in zip(hparams_encoder_static.vocab_sizes,hparams_encoder_static.embedding_dims):
            self.embeddings.append(nn.Embedding(v,e))
        self.input_dim = int(np.sum(hparams_encoder_static.embedding_dims) + hparams_input_shape.static_dim)
        self.input_channels = hparams_input_shape.static_dim + hparams_input_shape.static_dim_cat
class BasicEncoderStaticMLP(BasicEncoderStatic):
    def __init__(self, hparams_encoder_static, hparams_input_shape, target_dim=None):
        super().__init__(hparams_encoder_static, hparams_input_shape, target_dim)

        lin_ftrs = [self.input_dim] + list(hparams_encoder_static.lin_ftrs)
        if(target_dim is not None and lin_ftrs[-1]!=target_dim):
            lin_ftrs.append(target_dim)
        ps = [hparams_encoder_static.dropout] if not isinstance(hparams_encoder_static.dropout, Iterable) else hparams_encoder_static.dropout
        if len(ps)==1:
            ps= [ps[0]/2] * (len(lin_ftrs)-2) + ps
        actns = [nn.ReLU(inplace=True)] * (len(lin_ftrs)-2) + [None]
        layers = []
        for ni,no,p,actn in zip(lin_ftrs[:-1],lin_ftrs[1:],ps,actns):
            layers+=bn_drop_lin(ni,no,hparams_encoder_static.batch_norm,p,actn,layer_norm=False)
        self.layers=nn.Sequential(*layers)

        self.output_shape = dataclasses.replace(hparams_input_shape)
        self.output_shape.static_dim = int(lin_ftrs[-1])
        self.output_shape.static_dim_cat = 0

    def forward(self, static, static_cat=None):

        x = static #categorical branch
        if self.embeddings is not None and static_cat is not None:
            cat_out = [
            emb(static_cat[:, i])
            for i, emb in enumerate(self.embeddings)
            ]

        x_cat = torch.cat(cat_out, dim=1)
        x = torch.cat([x, x_cat], dim=1)

        return {"static": self.layers(x)}

    def get_output_shape(self):
        return self.output_shape

In [ ]:
# ----------------------------------------
# 5. Model initialization (BasicEncoderStaticMLP unchanged)
# ----------------------------------------
@dataclass
class MLPConfig:
    embedding_dims: List[int] = field(default_factory=lambda: [16, 16, 16])
    vocab_sizes: List[int]    = field(default_factory=lambda: [2, 5, 5])
    lin_ftrs: List[int]       = field(default_factory=lambda: [128, 128, 128])
    dropout: float = 0.5
    batch_norm: bool = True

@dataclass
class ShapeCfg:
    static_dim: int = 0
    static_dim_cat: int = 0
    channels: int = 0
    length: int = 0
    sequence_last: bool = False
    channels2: int = 0

shape = ShapeCfg(static_dim=len(cont_features), static_dim_cat=len(cat_features))

mlp_cfg = MLPConfig(
    embedding_dims=[unique_counts[c] for c in cat_features],
    vocab_sizes=[unique_counts[c] for c in cat_features],
    lin_ftrs=LIN_FTRS
)

encoder = BasicEncoderStaticMLP(mlp_cfg, shape, target_dim=len(target_cols))
print(f"\nModel parameters: {sum(p.numel() for p in encoder.parameters()):,}")


In [ ]:
# ----------------------------------------
# 6. Loss and Optimizer
# ----------------------------------------
# Binary Cross Entropy with NaN masking (ignore_nans=True behavior)
def bce_loss_with_nans(preds, targets):
    mask = ~torch.isnan(targets)
    if mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True)
    return nn.BCEWithLogitsLoss()(preds[mask], targets[mask])

optimizer = torch.optim.AdamW(encoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)